# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In the dataset one row means one content page, belong to one client, on one single day. For example, if a page was tracked for 31 days in March, then that page will have 31 separate rows one for each day.

The time window used is March 2026, a mid-panel month chosen on purpose. The final month (June 2026) was avoided to use because it's reserved as the future window for testing phase later. If i use it now it would mean expose the answer before doing anything.

I  verified it below with a code that checked the row count and the earliest or latest date in the data that confirm 9,841,378 rows spanning exactly March 1 to March 31, 2026.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (safe to use for prediction):

gsc_impressions =how many times the page appeared in search results that day
gsc_clicks =how many times people clicked on it that day
gsc_avg_position =where the page ranked in search results that day
ctr_calc (clicks ÷ impressions) =a simple derived rate from the two numbers above

Label (what I'd want to predict):

A future outcome whether a page's traffic or clicks decline over the next 30 days or not, built from dates after the feature window. This label look like classification-style (decline: yes/no, or a probability of decline), but it is used for final ranking/scoring output: pages are not simply labeled yes/no in isolation, they are ordered by their decline-risk score, so an editor can review the highest-risk pages first. The label is the building block; the ranked priority list is the actual deliverable.

Context:

content_hash_id, client_hash_id = these are just scrambled ID codes. They identify which page and client a row belongs to, but carry no real information themselves.

Excluded :

Any pre-built product score (like health_score or priority_score) = even though these aren't in this dataset, I'm noting I would never use them as a feature. Using a score the product already calculated would mean the model just copies an existing decision instead of learning something real from the raw data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This cell loads a secret access key (a "token") that I got from Hugging Face, so I can connect to the warehouse dataset later without ever typing the key directly into the notebook. The key is stored in a hidden .env file, and this code simply reads it from there. 

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
print("Token loaded:", hf_token is not None)

Token loaded: True


This cell starts up DuckDB, a tool that lets me run SQL queries directly on large remote datasets without downloading them first. It then registers my Hugging Face token with DuckDB. The print statement just confirms that both steps completed successfully.

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
print("DuckDB connected and secret registered")

DuckDB connected and secret registered


This cell points DuckDB at one month (March 2026) of the fact_content_daily_performance table It then runs a quick query to count how many rows are in that month and check the earliest and latest dates present. 

In [4]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

check = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM {REL}
""").df()

print(check)

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


 It verifies that each row really does represent one unique combination of date, client, and content page, with no  duplicates. It groups the data by date, client, and content, then looks for any group that appears more than once. An empty result means no duplicates were found.

In [5]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
    FROM {REL}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(grain_check)

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, cnt]
Index: []


This cell checks how many rows in the March 2026 data actually have real Google Analytics (GA4) data available, using the ga4_data_available flag. It counts the total number of rows, then separately counts only the rows where GA4 tracking was truly active. This matters because many rows have GA4 numbers filled in as zero simply because tracking wasn't set up yet, this check reveals how much of the data can be trusted for GA4-based analysis.

In [6]:
availability = con.sql(f"""
    SELECT COUNT(*) as total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available
    FROM {REL}
""").df()

print(availability)

     total  ga4_available
0  9841378       413966.0


This cell pulls just 5 sample rows from the dataset and prints out the names of all its columns. It's a quick way to see exactly what fields are available in the table.

In [8]:
schema = con.sql(f"""
    SELECT * FROM {REL} LIMIT 5
""").df()

print(schema.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


 It pulls 1,000 real rows from March 2026, keeping only rows where search data was genuinely tracked (gsc_data_available IS TRUE), and selects five columns: two ID columns (for identifying pages and clients) and three real search-performance signals (impressions, clicks, position). It also calculates a new column, ctr_calc, by dividing clicks by impressions to get the click-through rate..head()at the end just shows the first few rows so I can visually confirm the data looks correct.

In [9]:
features_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           CAST(gsc_clicks AS FLOAT) / NULLIF(gsc_impressions, 0) AS ctr_calc
    FROM {REL}
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()

features_df.head()

,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr_calc
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,20,0,3.350000,0.000
1,content_05597932fe4da067,client_73cda7b4e4f265ea,1,0,0.000000,0.000
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,125,1,4.928000,0.008
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,7,0,4.000000,0.000
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,11,0,2.272727,0.000


This cell demonstrates the leakage trap on purpose. First, it creates a simple target, whether a page has a "high CTR" (above 5%) and checks how strongly a real, honest feature (gsc_avg_position) relates to it, giving a low, realistic correlation. Then it deliberately creates a "cheating" feature built directly from the same value used to define the target itself, and checks its correlation which jumps up sharply, looking impressive but meaning nothing real, since it's just the answer in disguise. Finally, the cheating feature is removed, and the original honest, weaker correlation is shown again as the true, trustworthy result. This is meant to show, hands-on, why leaked features must never be trusted just because they produce a "good" score.

In [ ]:
import numpy as np

features_df["target_high_ctr"] = (features_df["ctr_calc"] > 0.05).astype(int)


honest_score = features_df["gsc_avg_position"].corr(features_df["target_high_ctr"])
print("Honest correlation (position vs target):", round(honest_score, 3))


features_df["cheating_feature"] = features_df["ctr_calc"] * 1.0  

cheat_score = features_df["cheating_feature"].corr(features_df["target_high_ctr"])
print("Cheating correlation (leaked feature vs target):", round(cheat_score, 3))


features_df = features_df.drop(columns=["cheating_feature"])
print("Leak removed. Honest score stands at:", round(honest_score, 3))

Honest correlation (position vs target): -0.011
Cheating correlation (leaked feature vs target): 0.644
Leak removed. Honest score stands at: -0.011


**The trap, demonstrated:** The honest feature (gsc_avg_position) shows almost no correlation 
with the proxy target (-0.011) = a realistic result. But when I deliberately 
added cheating_feature, built directly from the same value used to define the target 
(ctr_calc), the correlation jumped to 0.644 = a fake  result. 

This is fake progress: the model isn't learning any real pattern, it's just recognizing the 
label in disguise. Once the leaked feature was removed, the score returned to the honest, much 
weaker -0.011. This is exactly the leakage trap described in the lane guide a feature 
mathematically derived from the label will always look perfect and teach you nothing real. 
Any result built this way would collapse the moment it faced genuinely new data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has a few honest limits worth stating clearly:

1. Only one month of data. This slice covers just March 2026. It can't show seasonal changes (like holiday traffic spikes) or long-term trends, because it's a single snapshot in time, not a full year.

2. Most rows don't have Analytics (GA4) data. Checking with ga4_data_available IS TRUE showed that only about 4.2% of rows (413,966 out of 9,841,378) have real Analytics data. The rest have GA4 numbers filled in as zero but that zero doesn't mean "no visitors, it means "we weren't tracking this yet. Treating those zeros as real would be a mistake.

3. Not every client has the same amount of history. Some clients have been tracked for much longer than others. Comparing two clients directly without checking when each one's tracking started could unfairly make a newer client look worse, when really there's just less data for them.

4. This data only shows what already happened it can't prove cause and effect. Even if a page's traffic goes up after being refreshed, this data alone can't prove the refresh caused it. Proving that would need a proper experiment, which is outside what this dataset can do.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.